# Aims
- When querying the RAG database using TurboQuant, a lookup table is used to implement **Asymmetric Distance Computation**
- Basically, TurboQuant's centroids are per embedding coordinate and not across all coordinates
- Importantly the centroids are fixed (i.e. data agnostic)
- The thing is, if $b=4$, then I would have packed 2 coordinates into a single `uint8`
- The calculated lookup table needs to store the distances associated with any two consecuitive centroid assignments
- For example, if $b=4$ that means that there 16 possible centroids per coordinate. Which means that there are 256 possible combinations of 2 consecutive centroids. I.e. the lookup table needs to have 256 entries 

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pickle as pkl
import polars as pl
from sentence_transformers import SentenceTransformer

from lib.algorithms import fwht_batch
from lib.turboquant.turboquant import quantize_embeddings_numpy, pack_bits_numpy


In [2]:
with open("../data/codebook/384d_codebook.pkl", "rb") as f:
    codebook = pkl.load(f)

In [3]:
codebook

{'1bits': array([-0.04070159,  0.04088104]),
 '2bits': array([-0.07087235, -0.01866579,  0.02315479,  0.07451284]),
 '3bits': array([-0.09786132, -0.05691854, -0.03069189, -0.00993948,  0.01022873,
         0.03272187,  0.06077609,  0.10216256]),
 '4bits': array([-0.12091945, -0.08460474, -0.06134046, -0.04448182, -0.03125859,
        -0.01997488, -0.00952856,  0.00078499,  0.0112464 ,  0.02168263,
         0.0320071 ,  0.04226933,  0.05339511,  0.06735399,  0.08780586,
         0.12175898])}

In [4]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embed_dim = embedder.get_embedding_dimension()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
QUERY = "What is the social impact of war?"

In [6]:
query_emb = embedder.encode(QUERY, convert_to_numpy=True).astype(np.float32)[np.newaxis, :]

In [7]:
query_emb.shape

(1, 384)

In [8]:
rot_query_emb, query_d_signs = fwht_batch(query_emb, seed=42)

In [9]:
rot_query_emb.shape

(1, 512)

In [10]:
lut = []

for i in range(256):
    
    c_1_id, c2_id = np.uint8(i) >> 4, np.uint8(i) & 0x0F
    
    cluster_dists = []
    
    for j in range(rot_query_emb.shape[1] // 2):
    
        cluster_dists.append((rot_query_emb.flatten()[2*j] * codebook["4bits"][int(c_1_id)]) + (rot_query_emb.flatten()[2*j + 1] * codebook["4bits"][int(c2_id)]))
        
    lut.append(cluster_dists)

In [11]:
lut = np.array(lut)

In [12]:
lut.shape

(256, 256)

In [13]:
from lib.turboquant.turboquant import calculate_query_distance_lookup_table

In [14]:
lut2 = calculate_query_distance_lookup_table(rot_query_emb, 4, codebook)

In [15]:
lut2

array([[ 1.73845632e-03, -1.22430911e-03, -6.97484864e-03, ...,
        -8.25983832e-03,  9.25278853e-03,  1.68249146e-03],
       [ 2.98303685e-03, -3.75999372e-04, -6.47869451e-03, ...,
        -7.66418212e-03,  8.87873612e-03,  1.52201622e-03],
       [ 3.78035202e-03,  1.67452988e-04, -6.16084347e-03, ...,
        -7.28258710e-03,  8.63910706e-03,  1.41921103e-03],
       ...,
       [-3.61509849e-03, -3.80884677e-05,  6.27995969e-03, ...,
         7.42480163e-03, -8.75664223e-03, -1.45375641e-03],
       [-2.91417032e-03,  4.39666233e-04,  6.55938589e-03, ...,
         7.76026583e-03, -8.96730266e-03, -1.54413354e-03],
       [-1.75052631e-03,  1.23280941e-03,  7.02327454e-03, ...,
         8.31718582e-03, -9.31703001e-03, -1.69417289e-03]],
      shape=(256, 256))

In [16]:
lut

array([[ 1.73845632e-03, -1.22430911e-03, -6.97484864e-03, ...,
        -8.25983832e-03,  9.25278853e-03,  1.68249146e-03],
       [ 2.98303685e-03, -3.75999372e-04, -6.47869451e-03, ...,
        -7.66418212e-03,  8.87873612e-03,  1.52201622e-03],
       [ 3.78035202e-03,  1.67452988e-04, -6.16084347e-03, ...,
        -7.28258710e-03,  8.63910706e-03,  1.41921103e-03],
       ...,
       [-3.61509849e-03, -3.80884677e-05,  6.27995969e-03, ...,
         7.42480163e-03, -8.75664223e-03, -1.45375641e-03],
       [-2.91417032e-03,  4.39666233e-04,  6.55938589e-03, ...,
         7.76026583e-03, -8.96730266e-03, -1.54413354e-03],
       [-1.75052631e-03,  1.23280941e-03,  7.02327454e-03, ...,
         8.31718582e-03, -9.31703001e-03, -1.69417289e-03]],
      shape=(256, 256))

In [17]:
np.allclose(lut, lut2)

True

In [18]:
lut - lut2

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(256, 256))

Each row is a possible combination of 2 centroids.

Each column is the cosine distance between the 2 corresponding centroids with the rotated query embedding.

So for each document in the RAG database, for each coordinate in its embedding (which is 2 centroids packed into a single `uint8`), lookup the table and find the corresponding cosine distance and then sum across all the elements of the document embedding coordinates.

In [19]:
docs = pl.scan_parquet("../data/processed/arxiv_quantized_embeddings_4bits.parquet")

In [20]:
docs.columns

/var/folders/0m/rh70gs7d34l46d2kb6f6nzjr0000gn/T/ipykernel_3643/3265151099.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  docs.columns


['id', 'packed_embedding', 'embedding']

In [21]:
def l(x):
    x = x.to_numpy()

    return pl.Series(lut[x, np.arange(x.shape[1])].sum(axis=1))


In [22]:
sim_docs = docs.with_columns(
    pl.col("packed_embedding").map_batches(l, return_dtype=pl.Float64).alias("lut_dists")
).sort(pl.col("lut_dists"), descending=True).head(10).collect()

In [23]:
sim_docs["id"].to_list()

['0710.2140',
 '0704.0003',
 '0704.0004',
 '0704.0005',
 '0704.0006',
 '0704.0007',
 '0704.0008',
 '0704.0009',
 '0704.0010',
 '0704.0011']

In [24]:
raw_docs = pl.scan_parquet("../data/processed/arxiv_embeddings.parquet")

In [25]:
raw_docs.columns

/var/folders/0m/rh70gs7d34l46d2kb6f6nzjr0000gn/T/ipykernel_3643/1015414588.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  raw_docs.columns


['id', 'content', 'embedding']

In [26]:
select_docs = raw_docs.filter(pl.col("id").is_in(sim_docs["id"].to_list())).select("content").collect()

In [27]:
for c in select_docs["content"].to_list():
    t, a = c.split("|")
    print(t.strip())
    print("-"*len(t.strip()))
    print(a.strip())
    print("\n")

Title: The evolution of the Earth-Moon system based on the dark matter field
  fluid model
------------------------------------------------------------------------------------------
Abstract:   The evolution of Earth-Moon system is described by the dark matter field
fluid model proposed in the Meeting of Division of Particle and Field 2004,
American Physical Society. The current behavior of the Earth-Moon system agrees
with this model very well and the general pattern of the evolution of the
Moon-Earth system described by this model agrees with geological and fossil
evidence. The closest distance of the Moon to Earth was about 259000 km at 4.5
billion years ago, which is far beyond the Roche's limit. The result suggests
that the tidal friction may not be the primary cause for the evolution of the
Earth-Moon system. The average dark matter field fluid constant derived from
Earth-Moon system data is 4.39 x 10^(-22) s^(-1)m^(-1). This model predicts
that the Mars's rotation is also slowin